# 02 — Data Cleaning

Load the merged table, understand what's missing and why, then save a clean version to `data/processed/`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
from pathlib import Path

def find_project_root(start, depth=5):
    path = start.resolve()
    for _ in range(depth):
        if (path / "src").exists() and (path / "requirements.txt").exists():
            return path
        path = path.parent
    raise RuntimeError(f"Can't find project root from {start}")

PROJECT_ROOT  = find_project_root(Path.cwd())
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FIGURES_DIR   = PROJECT_ROOT / "outputs" / "figures"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(PROJECT_ROOT))
from src.data_loader import load_raw_tables, build_master_df, add_parsed_lap_times
from src.features import build_feature_set, FEATURE_COLUMNS, TARGET_COLUMN

plt.style.use("seaborn-v0_8-darkgrid")
pd.set_option("display.max_columns", 50)
print("Root:", PROJECT_ROOT)

## Load master dataframe

In [ ]:
tables = load_raw_tables()
df = build_master_df(tables)
df = add_parsed_lap_times(df)
print(df.shape)
df.head(3)

## Missing values

Not everything missing is a problem — a lot of it is expected.

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
summary = pd.DataFrame({"missing": missing, "pct": missing_pct})
summary[summary["missing"] > 0].sort_values("missing", ascending=False)

## position vs positionOrder

`position` is NaN for DNFs (crashes, retirements). `positionOrder` still ranks them at the back — that's what we use.

In [ ]:
# how many entries have no classified position (DNF/DSQ etc.)
dnf_count = df["position"].isna().sum()
total     = len(df)
print(f"DNFs / non-classified: {dnf_count} ({dnf_count/total:.1%} of all entries)")

# positionOrder is always set — confirm
print(f"positionOrder nulls: {df['positionOrder'].isna().sum()}")

## Status breakdown

See what actually caused all those non-finishes.

In [ ]:
status_counts = (
    df.merge(tables["status"], on="statusId", how="left")
    .groupby("status")
    .size()
    .sort_values(ascending=False)
    .head(20)
)
print(status_counts)

## Qualifying coverage by year

Q1/Q2/Q3 only exists from 2006. Before that there's a single session.

In [ ]:
q3_coverage = (
    df.groupby("year")["q3_seconds"]
    .apply(lambda x: x.notna().mean())
    .reset_index()
    .rename(columns={"q3_seconds": "q3_coverage"})
)

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(q3_coverage["year"], q3_coverage["q3_coverage"], color="steelblue", width=0.8)
ax.axvline(2006, color="red", linestyle="--", label="Q1/Q2/Q3 introduced")
ax.set_xlabel("Year")
ax.set_ylabel("Q3 data coverage")
ax.set_title("Q3 Qualifying Coverage by Year")
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "q3_coverage_by_year.png", dpi=150)
plt.show()

## Grid position distribution

Grid 0 means a pit-lane start — we recode that to 20 later.

In [ ]:
df["grid"] = pd.to_numeric(df["grid"], errors="coerce")
print("Grid value counts (top 25):")
print(df["grid"].value_counts().head(25))
print(f"\nGrid == 0 (pit lane starts): {(df['grid'] == 0).sum()}")

## Drop columns we won't use

Keep only what's relevant for modelling. Raw time strings (`q1`, `q2`, `q3`) are replaced by `*_seconds`.

In [ ]:
drop_cols = [
    "time",           # race finish time — only set for the winner
    "milliseconds",   # same info as time
    "fastestLap",     # lap number of fastest lap — not useful as feature
    "fastestLapTime", # raw string, not parsed
    "fastestLapSpeed",
    "q1", "q2", "q3", # replaced by q1_seconds etc.
    "number",         # car number, not predictive
]

# only drop columns that actually exist
drop_cols = [c for c in drop_cols if c in df.columns]
df_clean = df.drop(columns=drop_cols)
print(f"Dropped {len(drop_cols)} columns. Shape: {df_clean.shape}")

## Save cleaned dataframe

In [ ]:
out_path = PROCESSED_DIR / "master_cleaned.csv"
df_clean.to_csv(out_path, index=False)
print(f"Saved: {out_path}  ({df_clean.shape[0]} rows, {df_clean.shape[1]} cols)")